# Format Data and divide Dataframes

This Juypter Notebook was used to extract all needed data into usable dataframes, the data is not included in the repository

In [2]:
import numpy as np
import pandas as pd
import pymarc as pm
import os
import plotly.express as px
import utils
import xmltodict
import xml.etree.ElementTree as ET
import utils

In [ ]:
# 020 = ISBN
# 041 = Language Code
# 100 = Author Name
# 240 = Title
# 245 Title Statement a, b, c
# 264a Place of production, publication, distribution, manufacture
# 264b Name of producer, publisher, distributor, manufacturer
# 264c Date of production, publication, distribution, manufacture, or copyright notice 
# 300a Number of Pages
# 336b Content Type
# 084 where $2 is rvk Classification number of rvk
# 655a where $2 is gnd or gnd-content Genre
# 650a where $2 is gnd Topical term or geographic name entry element

In [ ]:
# make dictionary of MARC21 metadata fields to extract
fields = {
    'ISBN' : ['020', ['a'], []],
    'Language Code': ['041', ['a'], []],
    'Author': ['100', ['a'], []],
    'Title': ['240', ['a'], []],
    'Title Statement': ['245', ['a', 'b', 'c'], []],
    'Place': ['264', ['a'], []],
    'Publisher': ['264', ['b'], []],
    'Date' : ['264', ['c'], []],
    'Number of Pages': ['300', ['a'], []],
    'Content Type': ['336', ['b'], []], 
    'rvk Classification': ['084', ['a'], ['rvk', '']],
    'gnd Topical Term': ['650', ['a'], ['gnd', '']],
    'gnd Genre' : ['655', ['a'], ['gnd', 'gnd-content']]}
# iterate over all files in directory
rootdir = 'data/BVB'
for subdir, dirs, files in os.walk(rootdir):
    for i, file in enumerate(files):
        reader = pm.marcxml.parse_xml_to_array(os.path.join(subdir, file))
        # make dictionary from reader using selected metadata fields to convert to dataframe
        df_dict = {}
        for key, field in fields.items():
            temp = []
            for record in reader:
                temp.append(utils.get_record_metadata(record=record, field_no=field[0], subfields=field[1], restraint=field[2]))
            df_dict[key] = temp
        df = pd.DataFrame.from_dict(df_dict)
        # saves dictionary of respective file as .csv 
        df.to_csv('data/converted_data/b3kat_' + str(i), sep=';')
        # save memory
        del df
        del reader

In [ ]:
# merges all extracted .csv files into one dataframe and saves it 
rootdir = 'data/converted_data'
for subdir, dirs, files in os.walk(rootdir):
    for i, file in enumerate(files):
        if i == 0:
            df = pd.read_csv(os.path.join(subdir, file), sep=';', dtype=str)
        else:
            df_temp = pd.read_csv(os.path.join(subdir, file), sep=';', dtype=str)
            df = pd.concat([df, df_temp], ignore_index=True)
# drops old index
df = df.drop('Unnamed: 0', axis=1)
df.to_csv('data/converted_data/b3_complete', sep=';')

In [ ]:
# makes separate dataframe for records which have both rvk Classification and gnd Genre entry and saves as .csv
df_remove_na = df[df['rvk Classification'].notna()]
df_remove_na = df_remove_na[df_remove_na['gnd Genre'].notna()]
df_remove_na.to_csv('data/converted_data/b3_not_na', sep=';')